# Introduction
This notebook is used to fetch, plot, and analyze experiment results.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 26.4531


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package


## 2. Fetch Results

In [3]:
import seml
import pandas as pd

db_collection = 'llama-pert-awq-bnb-hqq'
states = ["COMPLETED"]

# Get the results
all_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)

print(f"Length of all_results before deduplication: {len(all_results)}")

# Get the list of columns that start with 'config.'
config_columns = [col for col in all_results.columns if col.startswith('config.')]

# Drop duplicates based on config columns, keeping the last occurrence
all_results = all_results.drop_duplicates(subset=config_columns, keep='last')

print(f"Length of all_results after deduplication: {len(all_results)}")
print("Columns used for deduplication:")
print(config_columns)
print("\nAll columns in the dataframe:")
print(all_results.columns)
all_results.head()

Output()

Output()

Length of all_results before deduplication: 4800
Length of all_results after deduplication: 4800
Columns used for deduplication:
['config.overwrite', 'config.db_collection', 'config.batch_size', 'config.dataset_name', 'config.device', 'config.exp_id', 'config.max_entries', 'config.max_new_tokens', 'config.model_name', 'config.n_beams', 'config.n_repeats', 'config.num_excel_rows', 'config.save_excel', 'config.seed', 'config.strategy', 'config.temperature', 'config.typo_intensity', 'config.typo_type', 'config.use_beam_search']

All columns in the dataframe:
Index(['_id', 'config.overwrite', 'config.db_collection', 'config.batch_size',
       'config.dataset_name', 'config.device', 'config.exp_id',
       'config.max_entries', 'config.max_new_tokens', 'config.model_name',
       'config.n_beams', 'config.n_repeats', 'config.num_excel_rows',
       'config.save_excel', 'config.seed', 'config.strategy',
       'config.temperature', 'config.typo_intensity', 'config.typo_type',
       'config

,_id,config.overwrite,config.db_collection,config.batch_size,config.dataset_name,config.device,config.exp_id,config.max_entries,config.max_new_tokens,config.model_name,...,result.Brier_adj,result.LogLoss_adj,result.Entropy_adj,result.AUCROC_sem,result.AUCPR_sem,result.Brier_sem,result.LogLoss_sem,result.Entropy_sem,result.Accuracy,result.fail_trace
0,1,1,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f5831a43be0>
1,2,2,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f5831a43be0>
2,3,3,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f5831a43be0>
3,4,4,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.579266,5.823508,45.919863,1.0,1.0,0.0,2.220446e-16,-3.765434e-08,0.375000,<function get_results at 0x7f5831a43be0>
4,5,5,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.624608,6.945512,41.225135,1.0,1.0,0.0,2.220446e-16,-3.246064e-08,0.323276,<function get_results at 0x7f5831a43be0>


### 2.1 Check Failed Rows

In [8]:
import seml
import pandas as pd
from collections import Counter

db_collection = 'llama-pert-awq-bnb-hqq'
states = ["FAILED"]

# Get the results
failed_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)
print(f"Length of failed_results before deduplication: {len(failed_results)}")

# Get the list of columns that start with 'config.'
config_columns = [col for col in failed_results.columns if col.startswith('config.')]

# Drop duplicates based on config columns, keeping the last occurrence
failed_results = failed_results.drop_duplicates(subset=config_columns, keep='last')
print(f"Length of failed_results after deduplication: {len(failed_results)}")

print("\nUnique values and their frequencies for each config parameter in failed_results:")
for col in ["config.dataset_name", "config.model_name", "config.typo_type", "config.typo_intensity"]:
    value_counts = Counter(failed_results[col])
    print(f"\n{col}:")
    for value, count in value_counts.items():
        print(f"  - {value}: {count}")

Output()

Output()

Length of failed_results before deduplication: 70
Length of failed_results after deduplication: 70

Unique values and their frequencies for each config parameter in failed_results:

config.dataset_name:
  - P364: 11
  - P37: 18
  - P740: 21
  - P101: 20

config.model_name:
  - Llama-3-8B-BNB-4bit-local: 50
  - Llama-3-8B-HQQ-mixed-local: 20

config.typo_type:
  - word_phrase_translation: 2
  - word_context_aware_insertion: 1
  - word_remove_punctuation: 5
  - word_keyword_only: 5
  - word_taxonomy_pos: 6
  - word_taxonomy_neg: 5
  - none: 5
  - char_insertion: 5
  - char_deletion: 6
  - char_replacement: 4
  - char_repetition: 5
  - char_swapping: 3
  - word_CMW: 4
  - char_LCC: 5
  - word_synonym: 1
  - char_insert_noise: 3
  - word_repeat: 1
  - char_substitution: 3
  - word_emoji: 1

config.typo_intensity:
  - 1: 24
  - 2: 23
  - 3: 23


### 2.2 Check high accuracy columns

In [9]:
# Filter rows where accuracy is higher than 0.6
high_accuracy_results = all_results[all_results['result.Accuracy'] > 0.6]

# Print the number of rows that meet this criteria
print(f"Number of rows with accuracy > 0.6: {len(high_accuracy_results)}")

# Display the first few rows of the filtered results
print(high_accuracy_results[['config.model_name', 'config.strategy', 'result.Accuracy']].head())

# Count the number of high accuracy rows for each unique model name
model_counts = high_accuracy_results['config.model_name'].value_counts()

print("\nNumber of high accuracy rows for each model:")
print(model_counts)

# Optional: Calculate and print the percentage of high accuracy rows for each model
total_rows = len(all_results)
model_percentages = (model_counts / total_rows * 100).round(2)

print("\nPercentage of high accuracy rows for each model:")
print(model_percentages)

Number of rows with accuracy > 0.6: 2656
   config.model_name    config.strategy  result.Accuracy
54        Llama-3-8B  Direct Completion         0.669540
55        Llama-3-8B  Direct Completion         0.744253
56        Llama-3-8B  Direct Completion         0.778736
60        Llama-3-8B  Direct Completion         0.939611
61        Llama-3-8B  Direct Completion         0.939611

Number of high accuracy rows for each model:
config.model_name
Llama-3-8B                    756
Llama-3-8B-AWQ-4bit-local     714
Llama-3-8B-BNB-4bit-local     620
Llama-3-8B-HQQ-mixed-local    566
Name: count, dtype: int64

Percentage of high accuracy rows for each model:
config.model_name
Llama-3-8B                    15.75
Llama-3-8B-AWQ-4bit-local     14.88
Llama-3-8B-BNB-4bit-local     12.92
Llama-3-8B-HQQ-mixed-local    11.79
Name: count, dtype: float64


## 3. Plot results

In [4]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from fpdf import FPDF
import numpy as np
from PIL import Image

# Add these imports at the top
import matplotlib.pyplot as plt

# import matplotlib
# matplotlib.rcParams['text.usetex'] = True

# Replace with:
import matplotlib
matplotlib.rcParams['text.usetex'] = False

# If you still see font-related issues, you might also want to add:
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

from src.models import MODEL_NUM_BITS

# Define color scheme for the four models
model_colors = {
    'Llama-3-8B': '#1f77b4',  # Blue
    'Llama-3-8B-AWQ-4bit-local': '#ff7f0e',  # Orange
    'Llama-3-8B-BNB-4bit-local': '#2ca02c',  # Green
    'Llama-3-8B-HQQ-mixed-local': '#d62728'  # Red
}

# Define color scheme for intensities
intensity_colors = {
    1: '#1f77b4',  # Blue
    2: '#2ca02c',  # Green
    3: '#d62728'   # Red
}

model_name_map = {
    'Llama-3-8B': 'Llama-3-8B-BF16',
    'Llama-3-8B-AWQ-4bit-local': 'Llama-3-8B-AWQ-4bit',
    'Llama-3-8B-BNB-4bit-local': 'Llama-3-8B-BNB-4bit',
    'Llama-3-8B-HQQ-mixed-local': 'Llama-3-8B-HQQ-3-4bit'
}

perturbations_to_plot = [
    'char_insertion',
    'char_deletion',
    'char_replacement',
    'char_repetition',
    'char_swapping',
    'char_LCC',
    'word_synonym',
    'char_insert_noise',
    'word_repeat',
    'char_substitution',
    'word_emoji',
    'word_internet_slang',
    'word_phrase_translation',
    'word_context_aware_insertion',
    'word_keyword_only',
]

perturbations_to_plot = ['none'] + sorted(perturbations_to_plot)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/fpdf/__init__.py:39: UserWarning: You have both PyFPDF & fpdf2 installed. Both packages cannot be installed at the same time as they share the same module namespace. To only keep fpdf2, run: pip uninstall --yes pypdf && pip install --upgrade fpdf2
  warnings.warn(


## 3.1 Radar plots

In [ ]:
from pathlib import Path


def create_single_radar_plot(df, metric, intensity, base_model, comparison_model, plot_title):
    """
    Creates a single radar plot comparing two models.
    """
    # Filter data for the given intensity
    df_intensity = df[df['config.typo_intensity'] == intensity]
    
    # Get baseline value
    baseline_value = df_intensity[
        (df_intensity['config.typo_type'] == 'none') &
        (df_intensity['config.model_name'] == base_model)
    ][f'result.{metric}'].mean()
    
    # Get all perturbation types
    angles = np.linspace(0, 2*np.pi, len(perturbations_to_plot), endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))
    baseline_values = np.full(len(perturbations_to_plot) + 1, baseline_value)
    
    # Create figure
    fig = go.Figure()
    
    # Get base model values
    base_values = []
    for pert_type in perturbations_to_plot:
        value = df_intensity[
            (df_intensity['config.typo_type'] == pert_type) &
            (df_intensity['config.model_name'] == base_model)
        ][f'result.{metric}'].mean()
        base_values.append(value)
    base_values = np.concatenate((base_values, [base_values[0]]))
    
    # Get comparison model values
    comp_values = []
    for pert_type in perturbations_to_plot:
        value = df_intensity[
            (df_intensity['config.typo_type'] == pert_type) &
            (df_intensity['config.model_name'] == comparison_model)
        ][f'result.{metric}'].mean()
        comp_values.append(value)
    comp_values = np.concatenate((comp_values, [comp_values[0]]))
    
    # Add baseline trace (dashed line)
    fig.add_trace(
        go.Scatterpolar(
            r=baseline_values,
            theta=np.concatenate((perturbations_to_plot, [perturbations_to_plot[0]])),
            name='Baseline (no typos)',
            line=dict(
                color='gray',
                dash='dash'
            ),
            showlegend=True
        )
    )
    
    # Add base model trace with fill
    fig.add_trace(
        go.Scatterpolar(
            r=base_values,
            theta=np.concatenate((perturbations_to_plot, [perturbations_to_plot[0]])),
            name=f'{model_name_map[base_model]} ({MODEL_NUM_BITS[base_model]}-bit)',
            line=dict(color=model_colors[base_model]),
            fillcolor=model_colors[base_model],
            fill='toself',
            opacity=0.3,
            showlegend=True
        )
    )
    
    # Add comparison model trace with fill
    fig.add_trace(
        go.Scatterpolar(
            r=comp_values,
            theta=np.concatenate((perturbations_to_plot, [perturbations_to_plot[0]])),
            name=f'{model_name_map[comparison_model]} ({MODEL_NUM_BITS[comparison_model]}-bit)' if comparison_model in ['Llama-3-8B'] else comparison_model,
            line=dict(color=model_colors[comparison_model]),
            fillcolor=model_colors[comparison_model],
            fill='toself',
            opacity=0.3,
            showlegend=True
        )
    )
    
    # Update layout
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1],
                tickfont=dict(size=12)
            ),
            angularaxis=dict(
                tickfont=dict(size=12)
            )
        ),
        showlegend=True,
        legend=dict(
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            x=0.5,
            orientation="h",
            font=dict(size=14)
        ),
        height=800,
        width=1000,
        title=dict(
            text=plot_title,
            font=dict(size=20),
            y=0.95,
            x=0.5,
            xanchor='center',
            yanchor='top'
        ),
        margin=dict(
            t=100,
            b=100,
            l=50,
            r=50
        )
    )
    
    return fig

def create_all_radar_plots(df, metric, intensity):
    """
    Creates three separate radar plots for different model comparisons.
    """
    # Define base model and comparison models
    base_model = 'Llama-3-8B'
    comparison_models = [
        'Llama-3-8B-BNB-4bit-local',
        'Llama-3-8B-AWQ-4bit-local',
        'Llama-3-8B-HQQ-mixed-local'
    ]
    plot_titles = [
        "Llama vs BNB",
        "Llama vs AWQ",
        "Llama vs HQQ"
    ]
    
    # Create and display each plot
    figures = []
    for comparison_model, plot_title in zip(comparison_models, plot_titles):
        fig = create_single_radar_plot(
            df, 
            metric, 
            intensity, 
            base_model, 
            comparison_model, 
            f"{plot_title} - {metric} with intensity {intensity}"
        )
        figures.append(fig)
        fig.show()
        
        # Create save directory if it doesn't exist
        save_dir = "plots"
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        
        # Save plots
        fig.write_image(save_dir / f"{metric.lower()}_{comparison_model}_{intensity}_radar_plot.png")
    
    return figures

# Create all plots
debug_figs = create_all_radar_plots(all_results, 'Accuracy', 1)

## 3.2 Box plots

In [9]:
from typing import Tuple, Dict, List
import plotly.graph_objects as go
import numpy as np
from pathlib import Path

def create_separate_box_and_bar_plots(
    df_filtered: pd.DataFrame,
    metric: str,
    save_dir: str = "plots",
    height: int = 600,
    width: int = 1200,
) -> Tuple[Dict[str, go.Figure], Dict[str, Dict[int, float]]]:
    """
    Creates two separate figures: a boxplot and a bar plot showing averages.
    Saves both plots as separate files.
    
    Args:
        df_filtered (pd.DataFrame): Filtered DataFrame containing the data
        metric (str): Metric to plot (e.g., 'Accuracy')
        save_dir (str): Directory to save the plots
        height (int): Height of each plot
        width (int): Width of each plot
    
    Returns:
        Tuple[Dict[str, go.Figure], Dict[str, Dict[int, float]]]: 
            - Dictionary containing both figure objects
            - Dictionary containing averaged values for each model and intensity
    """
    df_filtered = df_filtered[df_filtered['config.typo_type'].isin(perturbations_to_plot)]
    
    # Create separate figures
    fig_box = go.Figure()
    fig_bar = go.Figure()
    
    # Calculate baseline value
    baseline_value = df_filtered[
        (df_filtered['config.model_name'] == 'Llama-3-8B') &
        (df_filtered['config.typo_type'] == 'none')
    ][f'result.{metric}'].mean()
    
    # Get the x-axis range with padding
    x_min, x_max = 0.5, 3.5
    
    # Add baseline to both plots
    for fig in [fig_box, fig_bar]:
        fig.add_trace(
            go.Scatter(
                x=[x_min, x_max],
                y=[baseline_value, baseline_value],
                mode='lines',
                line=dict(color='gray', dash='dash', width=2),
                name='Baseline (no typos)',
                showlegend=True,
            )
        )
    
    models = df_filtered['config.model_name'].unique()
    intensities = [1, 2, 3]
    model_averages = {}  # Store averages for bar plot
    
    # For each model
    for i, model in enumerate(models):
        y_data = []
        x_data = []
        model_averages[model] = {}
        
        # For each intensity
        for intensity in intensities:
            df_model = df_filtered[
                (df_filtered['config.model_name'] == model) &
                (df_filtered['config.typo_intensity'] == intensity)
            ]
            
            # Aggregate results across perturbation types for each dataset
            datasets = df_model['config.dataset_name'].unique()
            aggregated_values = []
            
            for dataset in datasets:
                dataset_results = df_model[
                    df_model['config.dataset_name'] == dataset
                ][f'result.{metric}'].mean()
                aggregated_values.append(dataset_results)
            
            y_data.extend(aggregated_values)
            x_data.extend([str(intensity)] * len(aggregated_values))
            model_averages[model][intensity] = np.mean(aggregated_values)
        
        # Add boxplot trace
        fig_box.add_trace(
            go.Box(
                y=y_data,
                x=x_data,
                name=f'{model_name_map[model]}',
                marker_color=model_colors[model],
                showlegend=True,
                offsetgroup=str(i)
            )
        )
        
        # Add bar plot trace
        fig_bar.add_trace(
            go.Bar(
                x=[str(intensity) for intensity in intensities],
                y=[model_averages[model][intensity] for intensity in intensities],
                name=f'{model_name_map[model]}',
                marker_color=model_colors[model],
                showlegend=True,
                text=[f'{model_averages[model][intensity]:.3f}' for intensity in intensities],
                textposition='outside',
            )
        )
    
    # Common layout settings
    layout_common = dict(
        width=width,
        height=height,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            font=dict(size=14)
        ),
        plot_bgcolor='rgba(240, 244, 250, 1)',
        paper_bgcolor='white',
        margin=dict(t=100, b=50, l=50, r=50),
        xaxis=dict(
            title='Intensity',
            tickmode='array',
            tickvals=[1, 2, 3],
            ticktext=['1', '2', '3'],
            gridcolor='lightgray',
            showgrid=False,
        ),
        yaxis=dict(
            title=metric,
            range=[0, 1],
            gridcolor='lightgray',
            zerolinecolor='lightgray',
        )
    )
    
    # Update boxplot layout
    fig_box.update_layout(
        title=dict(
            text=f'Model Comparison: {metric} Distribution across Intensities',
            font=dict(size=16),
            y=0.95,
            x=0.5,
            xanchor='center',
            yanchor='top'
        ),
        boxmode='group',
        **layout_common
    )
    
    # Update bar plot layout
    fig_bar.update_layout(
        title=dict(
            text=f'Model Comparison: Average {metric} across Intensities',
            font=dict(size=16),
            y=0.95,
            x=0.5,
            xanchor='center',
            yanchor='top'
        ),
        **layout_common
    )
    
    # Create save directory if it doesn't exist
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Save plots
    fig_box.write_image(save_dir / f"{metric.lower()}_distribution_boxplot.png")
    fig_bar.write_image(save_dir / f"{metric.lower()}_averages_barplot.png")
    
    return {"boxplot": fig_box, "barplot": fig_bar}, model_averages

def create_and_show_separate_box_and_bar_plots(
    df_filtered: pd.DataFrame,
    metric: str,
    save_dir: str = "plots"
) -> Tuple[Dict[str, go.Figure], Dict]:
    """
    Creates, displays and saves the separate boxplot and bar plot visualizations.
    
    Args:
        df_filtered (pd.DataFrame): Filtered DataFrame containing the data
        metric (str): Metric to plot (e.g., 'Accuracy')
        save_dir (str): Directory to save the plots
    
    Returns:
        Tuple[Dict[str, go.Figure], Dict]: Dictionary of figure objects and dictionary of model averages
    """
    figs, model_averages = create_separate_box_and_bar_plots(df_filtered, metric, save_dir)
    figs["boxplot"].show()
    figs["barplot"].show()
    return figs, model_averages

# Example usage:
debug_figs, averages = create_and_show_separate_box_and_bar_plots(all_results, 'Accuracy')

## 3.3 Bar plots

In [13]:
def create_all_perturbations_bar_plot(df, metric, intensity):
    fig = go.Figure()
    
    # Get all unique models
    models = df['config.model_name'].unique()
    
    # Calculate the baseline values for each model
    baseline_values = {
        model: df[(df['config.typo_type'] == 'none') & 
                 (df['config.model_name'] == model)][f'result.{metric}'].mean()
        for model in models
    }
    
    # Get perturbation types excluding 'none'
    perturbation_types = [p for p in df['config.typo_type'].unique() if p != 'none']
    
    # Create bars for each model
    for i, model in enumerate(models):
        x = []
        for pert_type in perturbation_types:
            value = df[(df['config.typo_type'] == pert_type) &
                      (df['config.typo_intensity'] == intensity) &
                      (df['config.model_name'] == model)][f'result.{metric}'].mean()
            x.append(value)
            
        # Add vertical bars
        fig.add_trace(go.Bar(
            x=perturbation_types,  # Switched from y to x
            y=x,                   # Switched from x to y
            name=f'{model_name_map[model]} ({MODEL_NUM_BITS[model]}-bit)',
            marker_color=model_colors[model]
        ))
        
        # Add baseline line for each model
        fig.add_shape(
            type="line",
            x0=-0.5,              # Switched from y0 to x0
            x1=len(perturbation_types)-0.5,  # Switched from y1 to x1
            y0=baseline_values[model],        # Switched from x0 to y0
            y1=baseline_values[model],        # Switched from x1 to y1
            line=dict(
                color=model_colors[model],
                width=2,
                dash="dash"
            )
        )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f'Model Comparison: {metric} Bar Plots with intensity {intensity}',
            font=dict(size=16)
        ),
        xaxis_title="Perturbation Type",     # Switched from yaxis to xaxis
        yaxis_title=metric,                  # Switched from xaxis to yaxis
        barmode='group',
        height=800,                          # Increased height for better visibility
        width=1200,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            font=dict(size=14)
        ),
        font=dict(size=12)
    )
    
    # Create save directory if it doesn't exist
    save_dir = "plots"
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Save plots
    fig.write_image(save_dir / f"{metric.lower()}_{intensity}_all_bar_plot.png")
    
    return fig

debug_fig = create_all_perturbations_bar_plot(all_results, 'Accuracy', 1)
debug_fig.show()

## 3.4 PDF Visualization

### 3.4.1 Display debug plots

In [8]:
def display_debug_plots(figs, plot_titles, cols=3):
    # Filter figures and titles to only include those with "Accuracy"
    filtered_indices = [i for i, title in enumerate(plot_titles) if "Accuracy" in title]
    filtered_figs = [figs[i] for i in filtered_indices]
    filtered_titles = [plot_titles[i] for i in filtered_indices]
    
    # Calculate required number of rows based on filtered plots
    n_plots = len(filtered_figs)
    if n_plots == 0:
        print("No plots with 'Accuracy' found")
        return
        
    rows = int(np.ceil(n_plots / cols))
    
    # Create specs array for filtered plots
    specs = []
    for _ in range(rows):
        row_specs = []
        for _ in range(cols):
            row_specs.append({"type": "xy"})
        specs.append(row_specs)
    
    # Update specs based on actual figure types
    for i, fig in enumerate(filtered_figs):
        row = i // cols
        col = i % cols
        if hasattr(fig.data[0], 'type'):
            if fig.data[0].type == 'scatterpolar':
                specs[row][col] = {"type": "polar"}
            elif fig.data[0].type in ['bar', 'box']:
                specs[row][col] = {"type": "xy"}
        else:
            print(f"Warning: Could not determine type for figure {i}")
    
    # Create subplot figure with dynamic specs
    subplot_fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=filtered_titles,
        vertical_spacing=0.15,
        horizontal_spacing=0.05,
        specs=specs
    )
    
    # Add each filtered figure to the subplot
    for i, fig in enumerate(filtered_figs):
        row = (i // cols) + 1
        col = (i % cols) + 1
        for trace in fig.data:
            subplot_fig.add_trace(trace, row=row, col=col)
        subplot_fig.update_xaxes(title_text=fig.layout.xaxis.title.text, row=row, col=col)
        subplot_fig.update_yaxes(title_text=fig.layout.yaxis.title.text, row=row, col=col)
    
    # Update overall layout
    subplot_fig.update_layout(
        height=400 * rows,
        width=1800,
        showlegend=True,
        title_text="Debug View - Accuracy Plots",
    )
    subplot_fig.show()

### 3.4.2 Assemble PDF

In [11]:
from fpdf import FPDF
import matplotlib.pyplot as plt
import os
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

class MultipleVisualizationPDFGenerator:
    def __init__(self, plots_dir, exp_id, model_colors, model_name_map):
        self.plots_dir = plots_dir
        self.exp_id = exp_id
        self.model_colors = model_colors
        self.model_name_map = model_name_map
        self.debug_figs = []  # Store figures for debug display
        self.debug_titles = []  # Store titles for debug display
        
    class CustomPDF(FPDF):
        def __init__(self, metric):
            super().__init__()
            self.metric = metric
    
    def generate_config_pages(self, pdf, all_results, plot_type):
        # Add title page
        pdf.add_page()
        pdf.set_font("Helvetica", 'B', size=14)
        pdf.cell(0, 10, f"Model Comparison {plot_type.title()}s for {pdf.metric}", ln=True, align='C')
        
        config_columns = [
            "exp_id",
            "model_name",
            "dataset_name",
            "typo_type",
            "typo_intensity",
            "batch_size",
            "exp_id",
            "max_entries",
            "max_new_tokens",
            "n_beams",
            "n_repeats",
            "strategy",
            "temperature",
            "use_beam_search"
        ]
        
        # Separate configs into multiple and single value groups
        multiple_value_configs = []
        single_value_configs = []
        
        for col in config_columns:
            unique_values = all_results[f"config.{col}"].unique()
            if len(unique_values) > 1:
                multiple_value_configs.append((col, unique_values))
            else:
                single_value_configs.append((col, unique_values))
        
        # Print multiple value configs
        pdf.set_font("Helvetica", 'B', size=12)
        pdf.cell(0, 10, "Grid Configuration Parameters", ln=True)
        pdf.set_font("Helvetica", size=10)
        pdf.set_left_margin(10)
        
        for col, unique_values in multiple_value_configs:
            text = f"{col}: {', '.join(map(str, unique_values))}"
            pdf.set_x(10)
            pdf.multi_cell(0, 5, text)
        
        # Add some spacing between sections
        pdf.cell(0, 5, "", ln=True)
        
        # Print single value configs
        pdf.set_font("Helvetica", 'B', size=12)
        pdf.cell(0, 10, "Fixed Configuration Parameters", ln=True)
        pdf.set_font("Helvetica", size=10)
        
        for col, unique_values in single_value_configs:
            # Increase the width of the first cell from 30 to 50
            pdf.set_font("Helvetica", 'B', size=10)
            pdf.cell(50, 5, f"{col}: ", 0, 0)  # Changed from 30 to 50
            # Add remaining width calculation
            remaining_width = pdf.w - 60  # Account for margins
            pdf.set_font("Helvetica", '', size=10)
            pdf.multi_cell(remaining_width, 5, f"{', '.join(map(str, unique_values))}")

    def create_pdf_for_plots(self, plots, metric, plot_type, all_results):
        pdf = self.CustomPDF(metric)
        pdf.set_auto_page_break(auto=True, margin=25)
        
        self.generate_config_pages(pdf, all_results, plot_type)
        
        for plot_file, desc in plots:
            pdf.add_page()
            pdf.set_font("Helvetica", 'B', size=16)  # Increased font size for descriptions
            pdf.cell(0, 20, desc, ln=True, align='C')
            
            with Image.open(plot_file) as img:
                img_width, img_height = img.size
            
            scale_factor = (pdf.w - 20) / img_width
            scaled_height = img_height * scale_factor
            
            pdf.image(plot_file, x=10, y=pdf.get_y(), w=pdf.w-20, h=scaled_height)
        
        output_path = os.path.join(
            self.plots_dir, 
            f"model_comparison_{metric}_{plot_type}_{self.exp_id}.pdf"
        )
        pdf.output(output_path)
        print(f"Generated PDF: {output_path}")

    def generate_radar_plots(self, all_results, metric):
        plots = []
        debug_figs = []
        debug_titles = []
        
        for intensity in [1, 2, 3]:
            fig = create_all_radar_plots(all_results, metric, intensity)
            plot_file = os.path.join(self.plots_dir, f"radar_plot_{metric}_intensity_{intensity}.png")
            fig.write_image(plot_file)
            plots.append((plot_file, f"Radar Plot - Intensity {intensity}"))
            
            # Store for debug display
            debug_figs.append(fig)
            debug_titles.append(f"Radar Plot - {metric} - Intensity {intensity}")
            
        self.debug_figs.extend(debug_figs)
        self.debug_titles.extend(debug_titles)
        return [("radar", metric, plots)]

    def generate_box_plots(self, all_results, metric):
        plots = []
        fig = create_and_show_separate_box_and_bar_plots(all_results, metric)
        plot_file = os.path.join(self.plots_dir, f"box_plot_{metric}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"Box Plot Distribution"))
        
        # Store for debug display
        self.debug_figs.append(fig)
        self.debug_titles.append(f"Box Plot - {metric}")
        
        return [("boxplot", metric, plots)]

    def generate_bar_plots(self, all_results, metric):
        plots = []
        debug_figs = []
        debug_titles = []
        
        for intensity in [1, 2, 3]:
            fig = create_all_perturbations_bar_plot(all_results, metric, intensity)
            plot_file = os.path.join(self.plots_dir, f"bar_plot_{metric}_intensity_{intensity}.png")
            fig.write_image(plot_file)
            plots.append((plot_file, f"Bar Plot - Intensity {intensity}"))
            
            # Store for debug display
            debug_figs.append(fig)
            debug_titles.append(f"Bar Plot - {metric} - Intensity {intensity}")
            
        self.debug_figs.extend(debug_figs)
        self.debug_titles.extend(debug_titles)
        return [("barplot", metric, plots)]
    
    def generate_all_pdfs(self, all_results, metrics):
        """
        Generate PDFs for all visualization types for each metric
        
        Parameters:
        all_results (DataFrame): The complete results DataFrame
        metrics (list): List of metrics to generate visualizations for
        """
        for metric in metrics:
            # Reset debug collections for each metric
            self.debug_figs = []
            self.debug_titles = []
            
            # Generate all plot types
            radar_plots = self.generate_radar_plots(all_results, metric)
            box_plots = self.generate_box_plots(all_results, metric)
            bar_plots = self.generate_bar_plots(all_results, metric)
            
            # Display all debug plots for this metric
            display_debug_plots(self.debug_figs, self.debug_titles)
            
            # Combine all plots
            all_plot_data = radar_plots + box_plots + bar_plots
            
            # Create PDFs for each plot type
            for plot_type, metric_name, plots in all_plot_data:
                self.create_pdf_for_plots(plots, metric_name, plot_type, all_results)


if __name__ == "__main__":
    # Specify the experiment ID
    exp_id = "awq_hqq_bnb_comparison-10-31"
    
    # Create plots directory
    plots_dir = f"plots/model_comparison_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)
    
    # Initialize PDF generator
    pdf_generator = MultipleVisualizationPDFGenerator(
        plots_dir, 
        exp_id, 
        model_colors, 
        model_name_map
    )
    
    # Generate PDFs for different metrics
    metrics = ["Accuracy", "AUCPR_sample"]  # Add other metrics as needed
    pdf_generator.generate_all_pdfs(all_results, metrics)

AttributeError: 'list' object has no attribute 'write_image'

## 4. Debug HQQ

In [8]:
import pandas as pd
import numpy as np

# Parameters for filtering
metric = 'Accuracy'  # or 'AUCPR_sample'
model = 'Llama-3-8B-HQQ-mixed-local'  # adjust as needed

# Get the exact filtered DataFrame we're interested in
filter_condition = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == model)
filtered_df = all_results[filter_condition]

# Print information about the filtering
print(f"\nFiltering for:")
print(f"config.typo_type == 'none' AND config.model_name == '{model}'")

print(f"\nNumber of rows found: {len(filtered_df)}")

# Print the complete filtered DataFrame
print("\nComplete filtered DataFrame:")
print(filtered_df)

# Print the specific columns we're most interested in
columns_of_interest = ['config.typo_type', 'config.model_name', f'result.{metric}']
print(f"\nFiltered DataFrame (key columns only):")
print(filtered_df[columns_of_interest])

# Print the actual baseline value that would be used
if not filtered_df.empty:
    baseline_value = filtered_df[f'result.{metric}'].iloc[0]
    print(f"\nBaseline value that would be used: {baseline_value}")
    
    # Additional validation
    if len(filtered_df) > 1:
        print("\nWARNING: Multiple rows found! All values:")
        print(filtered_df[f'result.{metric}'].values)
else:
    print(f"\nWARNING: No data found for model '{model}' with typo_type 'none'")

# Print unique values in key columns to help with debugging
print("\nUnique values in key columns:")
print("\nUnique typo_types:")
print(all_results['config.typo_type'].unique())
print("\nUnique model_names:")
print(all_results['config.model_name'].unique())


Filtering for:
config.typo_type == 'none' AND config.model_name == 'Llama-3-8B-HQQ-mixed-local'

Number of rows found: 60

Complete filtered DataFrame:
       _id  config.overwrite    config.db_collection  config.batch_size  \
3550  3603              3603  llama-pert-awq-bnb-hqq                 32   
3590  3661              3661  llama-pert-awq-bnb-hqq                 32   
3591  3662              3662  llama-pert-awq-bnb-hqq                 32   
3592  3663              3663  llama-pert-awq-bnb-hqq                 32   
3650  3721              3721  llama-pert-awq-bnb-hqq                 32   
3651  3722              3722  llama-pert-awq-bnb-hqq                 32   
3652  3723              3723  llama-pert-awq-bnb-hqq                 32   
3710  3781              3781  llama-pert-awq-bnb-hqq                 32   
3711  3782              3782  llama-pert-awq-bnb-hqq                 32   
3712  3783              3783  llama-pert-awq-bnb-hqq                 32   
3770  3841            

In [10]:
all_results["config.model_name"].unique()

array(['Llama-3-8B', 'Llama-3-8B-AWQ-4bit-local',
       'Llama-3-8B-BNB-4bit-local', 'Llama-3-8B-HQQ-mixed-local'],
      dtype=object)

In [9]:
import pandas as pd
import numpy as np

# Models to compare
hqq_model = 'Llama-3-8B-HQQ-mixed-local'
base_model = 'Llama-3-8B'
metric = 'Accuracy'

# Get filtered DataFrames for both models
hqq_filter = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == hqq_model)
base_filter = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == base_model)

hqq_df = all_results[hqq_filter]
base_df = all_results[base_filter]

# Print side-by-side comparison
print("\n=== Model Comparison (Baseline Accuracy) ===")
print("-" * 50)
print(f"HQQ Model: {hqq_model}")
print(f"Base Model: {base_model}")
print("-" * 50)

if not hqq_df.empty and not base_df.empty:
    hqq_accuracy = hqq_df[f'result.{metric}'].iloc[0]
    base_accuracy = base_df[f'result.{metric}'].iloc[0]
    
    print(f"\nAccuracy Values:")
    print(f"{'Model':<30} {'Accuracy':<10}")
    print("-" * 40)
    print(f"{hqq_model:<30} {hqq_accuracy:.4f}")
    print(f"{base_model:<30} {base_accuracy:.4f}")
    
    # Calculate difference
    diff = hqq_accuracy - base_accuracy
    print(f"\nDifference (HQQ - Base): {diff:.4f}")
    print(f"Relative Change: {(diff/base_accuracy)*100:.2f}%")
    
    # Print warning if multiple rows found
    if len(hqq_df) > 1:
        print(f"\nWARNING: Multiple rows found for HQQ model! All values:")
        print(hqq_df[f'result.{metric}'].values)
    if len(base_df) > 1:
        print(f"\nWARNING: Multiple rows found for Base model! All values:")
        print(base_df[f'result.{metric}'].values)
else:
    if hqq_df.empty:
        print(f"No data found for HQQ model with typo_type 'none'")
    if base_df.empty:
        print(f"No data found for Base model with typo_type 'none'")

# Print full filtered DataFrames for verification
print("\n=== Full Filtered DataFrames ===")
print("\nHQQ Model DataFrame:")
print(hqq_df[['config.typo_type', 'config.model_name', f'result.{metric}']])
print("\nBase Model DataFrame:")
print(base_df[['config.typo_type', 'config.model_name', f'result.{metric}']])


=== Model Comparison (Baseline Accuracy) ===
--------------------------------------------------
HQQ Model: Llama-3-8B-HQQ-mixed-local
Base Model: Llama-3-8B
--------------------------------------------------

Accuracy Values:
Model                          Accuracy  
----------------------------------------
Llama-3-8B-HQQ-mixed-local     0.3448
Llama-3-8B                     0.3966

Difference (HQQ - Base): -0.0517
Relative Change: -13.04%

[0.34482759 0.87819857 0.87819857 0.87819857 0.4386423  0.4386423
 0.4386423  0.7026087  0.7026087  0.7026087  0.71794872 0.72649573
 0.72649573 0.80701754 0.80701754 0.80701754 0.67321613 0.67218201
 0.67218201 0.6827957  0.6827957  0.6827957  0.92464358 0.92464358
 0.92464358 0.90709459 0.90709459 0.90709459 0.25529661 0.25529661
 0.25529661 0.36621196 0.36621196 0.36621196 0.29370629 0.3030303
 0.3030303  0.71221532 0.71221532 0.71221532 0.6903024  0.6903024
 0.6903024  0.77128205 0.77128205 0.77128205 0.64953271 0.64953271
 0.64953271 0.7049689

## 5. Remove Duplicates

In [5]:
import pandas as pd

def inspect_and_remove_duplicates(df):
    # Define the columns used for pivoting
    pivot_columns = ['config.typo_type', 'config.typo_intensity']
    
    # Find duplicates in pivot columns
    duplicate_mask = df.duplicated(subset=pivot_columns, keep=False)
    duplicates = df[duplicate_mask]
    
    if duplicates.empty:
        print("No duplicates found in pivot columns.")
        return df
    
    print("Duplicate entries found in pivot columns:")
    print(duplicates[pivot_columns])
    
    print("\nFull rows for duplicate entries:")
    print(duplicates)
    
    # Ask user how to handle duplicates
    print("\nHow would you like to handle these duplicates?")
    print("1: Keep first occurrence")
    print("2: Keep last occurrence")
    print("3: Remove all duplicates")
    print("4: Do nothing (keep all)")
    
    choice = input("Enter your choice (1-4): ")
    
    if choice == '1':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='first')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '2':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='last')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '3':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep=False)
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '4':
        df_cleaned = df
        print("No rows removed.")
    else:
        print("Invalid choice. No rows removed.")
        df_cleaned = df
    
    return df_cleaned

# Assuming your dataframe is named 'all_results'
all_results_llama = inspect_and_remove_duplicates(all_results_llama)

# You can now use all_results_cleaned for further processing

Duplicate entries found in pivot columns:
           config.typo_type  config.typo_intensity
42  word_phrase_translation                      1
43  word_phrase_translation                      2
59  word_phrase_translation                      1
60  word_phrase_translation                      2

Full rows for duplicate entries:
    _id  config.overwrite config.db_collection config.dataset_name  \
42   43                43      llama-typo-eval                 P17   
43   44                44      llama-typo-eval                 P17   
59   61                61      llama-typo-eval                 P17   
60   62                62      llama-typo-eval                 P17   

   config.device    config.exp_id config.max_entries  config.max_new_tokens  \
42          cuda  typo-test-10-01               None                     25   
43          cuda  typo-test-10-01               None                     25   
59          cuda  typo-test-10-01               None                     25   
60